In [11]:
import pandas as pd
import numpy as np

import joblib
import os

from sklearn.model_selection import train_test_split

In [12]:
df = pd.read_csv("../data/application_train.csv")

xgb_model = joblib.load("../outputs/xgboost_model.pkl")

In [13]:
y = df["TARGET"]

X = df.drop(columns=["TARGET", "SK_ID_CURR"])


In [14]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_test.shape

(61503, 120)

In [25]:
y_proba = xgb_model.predict_proba(X_test)[:, 1]

finance_df = X_test.copy()

finance_df["actual_default"] = y_test.values
finance_df["default_probability"] = y_proba

In [26]:
LGD = 0.45

finance_df["EAD"] = finance_df["AMT_CREDIT"]

finance_df["expected_loss"] = (
    finance_df["default_probability"] * LGD * finance_df["EAD"]
)

finance_df[[
    "default_probability",
    "AMT_CREDIT",
    "EAD",
    "expected_loss"
]].head(10)

,default_probability,AMT_CREDIT,EAD,expected_loss
256571,0.382572,770292.0,770292.0,132611.404335
191493,0.308149,364896.0,364896.0,50598.996417
103497,0.793484,284400.0,284400.0,101550.072026
130646,0.417316,976711.5,976711.5,183418.667306
211898,0.476246,323194.5,323194.5,69264.017281
130549,0.675196,318528.0,318528.0,96780.981565
287975,0.161832,1431000.0,1431000.0,104211.924188
144906,0.110645,382500.0,382500.0,19044.836937
102173,0.834880,170640.0,170640.0,64108.776197
254495,0.507426,225000.0,225000.0,51376.872137


In [27]:
def probability_to_score(probability, min_score=300, max_score=850):
    score = max_score - probability * (max_score - min_score)
    return np.round(score).astype(int)

In [31]:
finance_df["credit_score"] = probability_to_score(
    finance_df["default_probability"]
)

finance_df[[
    "default_probability",
    "credit_score",
    "expected_loss"
]].head(10)

,default_probability,credit_score,expected_loss
256571,0.382572,640,132611.404335
191493,0.308149,681,50598.996417
103497,0.793484,414,101550.072026
130646,0.417316,620,183418.667306
211898,0.476246,588,69264.017281
130549,0.675196,479,96780.981565
287975,0.161832,761,104211.924188
144906,0.110645,789,19044.836937
102173,0.834880,391,64108.776197
254495,0.507426,571,51376.872137


In [42]:
def threshold_decision(probability):
    if probability <= 0.30:
        return "Approve"
    elif probability >= 0.50:
        return "Reject"
    else:
        return "Manual Review"

In [43]:
finance_df["credit_decision"] = finance_df["default_probability"].apply(
    threshold_decision
)

finance_df[[
    "actual_default",
    "default_probability",
    "credit_score",
    "AMT_CREDIT",
    "expected_loss",
    "credit_decision"
]].head(20)

,actual_default,default_probability,credit_score,AMT_CREDIT,expected_loss,credit_decision
256571,0,0.382572,640,770292.0,132611.404335,Manual Review
191493,0,0.308149,681,364896.0,50598.996417,Manual Review
103497,0,0.793484,414,284400.0,101550.072026,Reject
130646,0,0.417316,620,976711.5,183418.667306,Manual Review
211898,0,0.476246,588,323194.5,69264.017281,Manual Review
130549,0,0.675196,479,318528.0,96780.981565,Reject
287975,0,0.161832,761,1431000.0,104211.924188,Approve
144906,0,0.110645,789,382500.0,19044.836937,Approve
102173,0,0.834880,391,170640.0,64108.776197,Reject
254495,1,0.507426,571,225000.0,51376.872137,Reject


In [44]:
interest_rate = 0.12

finance_df["expected_interest"] = finance_df["AMT_CREDIT"] * interest_rate

finance_df["expected_profit"] = (
    finance_df["expected_interest"] - finance_df["expected_loss"]
)

finance_df[[
    "default_probability",
    "AMT_CREDIT",
    "expected_interest",
    "expected_loss",
    "expected_profit",
    "credit_decision"
]].head(20)

,default_probability,AMT_CREDIT,expected_interest,expected_loss,expected_profit,credit_decision
256571,0.382572,770292.0,92435.04,132611.404335,-40176.364335,Manual Review
191493,0.308149,364896.0,43787.52,50598.996417,-6811.476417,Manual Review
103497,0.793484,284400.0,34128.00,101550.072026,-67422.072026,Reject
130646,0.417316,976711.5,117205.38,183418.667306,-66213.287306,Manual Review
211898,0.476246,323194.5,38783.34,69264.017281,-30480.677281,Manual Review
130549,0.675196,318528.0,38223.36,96780.981565,-58557.621565,Reject
287975,0.161832,1431000.0,171720.00,104211.924188,67508.075812,Approve
144906,0.110645,382500.0,45900.00,19044.836937,26855.163063,Approve
102173,0.834880,170640.0,20476.80,64108.776197,-43631.976197,Reject
254495,0.507426,225000.0,27000.00,51376.872137,-24376.872137,Reject


In [45]:
decision_financial_analysis = finance_df.groupby("credit_decision").agg(
    applicants=("actual_default", "count"),
    defaults=("actual_default", "sum"),
    default_rate=("actual_default", "mean"),
    total_credit_exposure=("AMT_CREDIT", "sum"),
    avg_default_probability=("default_probability", "mean"),
    avg_credit_score=("credit_score", "mean"),
    total_expected_loss=("expected_loss", "sum"),
    avg_expected_loss=("expected_loss", "mean"),
    total_expected_interest=("expected_interest", "sum"),
    total_expected_profit=("expected_profit", "sum"),
    avg_expected_profit=("expected_profit", "mean")
).reset_index()

decision_financial_analysis

,credit_decision,applicants,defaults,default_rate,total_credit_exposure,avg_default_probability,avg_credit_score,total_expected_loss,avg_expected_loss,total_expected_interest,total_expected_profit,avg_expected_profit
0,Approve,21276,468,0.021997,1.371516e+10,0.201885,738.963386,1.238554e+09,58213.650449,1.645819e+09,4.072656e+08,19142.020820
1,Manual Review,19985,1099,0.054991,1.211004e+10,0.395767,632.330698,2.145983e+09,107379.669347,1.453205e+09,-6.927776e+08,-34664.876674
2,Reject,20242,3398,0.167869,1.093988e+10,0.647922,493.642031,3.165418e+09,156378.728453,1.312785e+09,-1.852633e+09,-91524.206967


In [46]:
decision_order = ["Approve", "Manual Review", "Reject"]

decision_financial_analysis["credit_decision"] = pd.Categorical(
    decision_financial_analysis["credit_decision"],
    categories=decision_order,
    ordered=True
)

decision_financial_analysis = decision_financial_analysis.sort_values(
    "credit_decision"
)

decision_financial_analysis

,credit_decision,applicants,defaults,default_rate,total_credit_exposure,avg_default_probability,avg_credit_score,total_expected_loss,avg_expected_loss,total_expected_interest,total_expected_profit,avg_expected_profit
0,Approve,21276,468,0.021997,1.371516e+10,0.201885,738.963386,1.238554e+09,58213.650449,1.645819e+09,4.072656e+08,19142.020820
1,Manual Review,19985,1099,0.054991,1.211004e+10,0.395767,632.330698,2.145983e+09,107379.669347,1.453205e+09,-6.927776e+08,-34664.876674
2,Reject,20242,3398,0.167869,1.093988e+10,0.647922,493.642031,3.165418e+09,156378.728453,1.312785e+09,-1.852633e+09,-91524.206967


In [47]:
approved_df = finance_df[finance_df["credit_decision"] == "Approve"]

approved_summary = {
    "approved_applicants": len(approved_df),
    "approved_default_rate": approved_df["actual_default"].mean(),
    "total_approved_credit": approved_df["AMT_CREDIT"].sum(),
    "total_expected_interest": approved_df["expected_interest"].sum(),
    "total_expected_loss": approved_df["expected_loss"].sum(),
    "total_expected_profit": approved_df["expected_profit"].sum(),
    "avg_expected_profit_per_approved_applicant": approved_df["expected_profit"].mean()
}

approved_summary

{'approved_applicants': 21276,
 'approved_default_rate': np.float64(0.021996615905245348),
 'total_approved_credit': np.float64(13715160516.0),
 'total_expected_interest': np.float64(1645819261.92),
 'total_expected_loss': np.float64(1238553626.954161),
 'total_expected_profit': np.float64(407265634.965839),
 'avg_expected_profit_per_approved_applicant': np.float64(19142.020819977395)}

In [48]:
interest_rates = [0.08, 0.10, 0.12, 0.15, 0.20]

sensitivity_results = []

approved_df = finance_df[finance_df["credit_decision"] == "Approve"].copy()

for rate in interest_rates:
    temp_expected_interest = approved_df["AMT_CREDIT"] * rate
    temp_expected_profit = temp_expected_interest - approved_df["expected_loss"]
    
    sensitivity_results.append({
        "interest_rate": rate,
        "approved_applicants": len(approved_df),
        "approved_default_rate": approved_df["actual_default"].mean(),
        "total_approved_credit": approved_df["AMT_CREDIT"].sum(),
        "total_expected_interest": temp_expected_interest.sum(),
        "total_expected_loss": approved_df["expected_loss"].sum(),
        "total_expected_profit": temp_expected_profit.sum(),
        "avg_expected_profit_per_applicant": temp_expected_profit.mean()
    })

sensitivity_df = pd.DataFrame(sensitivity_results)

sensitivity_df

,interest_rate,approved_applicants,approved_default_rate,total_approved_credit,total_expected_interest,total_expected_loss,total_expected_profit,avg_expected_profit_per_applicant
0,0.08,21276,0.021997,1.371516e+10,1.097213e+09,1.238554e+09,-1.413408e+08,-6643.202936
1,0.10,21276,0.021997,1.371516e+10,1.371516e+09,1.238554e+09,1.329624e+08,6249.408942
2,0.12,21276,0.021997,1.371516e+10,1.645819e+09,1.238554e+09,4.072656e+08,19142.020820
3,0.15,21276,0.021997,1.371516e+10,2.057274e+09,1.238554e+09,8.187205e+08,38480.938637
4,0.20,21276,0.021997,1.371516e+10,2.743032e+09,1.238554e+09,1.504478e+09,70712.468333


In [49]:
os.makedirs("../outputs", exist_ok=True)

finance_df[[
    "actual_default",
    "default_probability",
    "credit_score",
    "credit_decision",
    "AMT_CREDIT",
    "AMT_INCOME_TOTAL",
    "EAD",
    "expected_loss",
    "expected_interest",
    "expected_profit"
]].to_csv("../outputs/expected_loss_profit_results.csv", index=False)

decision_financial_analysis.to_csv(
    "../outputs/decision_financial_analysis.csv",
    index=False
)

sensitivity_df.to_csv(
    "../outputs/profit_sensitivity_analysis.csv",
    index=False
)

In [50]:
os.path.exists("../outputs/expected_loss_profit_results.csv")

True

## Interest rate sensitivity analysis

The sensitivity analysis evaluates how expected portfolio profit changes under different interest rate assumptions.

Only automatically approved applicants were included in this simulation, because rejected and manual review applicants are not assumed to generate immediate credit revenue.

At an 8% interest rate, the approved portfolio produces negative expected profit, meaning that expected losses exceed expected interest. Starting at a 10% interest rate, the portfolio becomes profitable under the current assumptions. As the interest rate increases, expected profit also increases.

This analysis shows how the Credit Intelligence Engine can support business decisions by connecting credit risk, approval policy, expected loss, and pricing assumptions.